In [1]:
!pip install gradio

In [5]:
import pandas as pd
import numpy as np
import gradio as gr
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

In [6]:
df = pd.read_csv("vehicle_data_with_final_price.csv")  #Load Dataset
df.dropna(subset=["ex_showroom_price", "on_road_price", "discount_percentage"], inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)  # Clean Data

categorical_cols = ["vehicle_type", "brand", "model", "fuel_type", "location"]
encoders = {}
original_values = {}

for col in categorical_cols:
    df[col] = df[col].astype(str)
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col])
    encoders[col] = encoder
    original_values[col] = sorted(encoder.classes_)

In [7]:
features = ["vehicle_type", "brand", "model", "fuel_type", "location", "year", "month"]
targets = ["ex_showroom_price", "on_road_price", "discount_percentage"]

models = {}
for target in targets:
    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        max_depth=12,
        min_samples_split=5,
        n_jobs=-1
    )
    model.fit(df[features], df[target])
    models[target] = model # Training model



In [12]:
def predict_prices(vehicle_type, brand, model_name, fuel_type, location, budget, target_year):

    def encode(col, val):
        try:
            return encoders[col].transform([val])[0]
        except:
            return 0

    vtype = encode("vehicle_type", vehicle_type)
    br = encode("brand", brand)
    mdl = encode("model", model_name)
    fuel = encode("fuel_type", fuel_type)
    loc = encode("location", location)

    months = [
        ("Jan", 1), ("Feb", 2), ("Mar", 3), ("Apr", 4),
        ("May", 5), ("Jun", 6), ("Jul", 7), ("Aug", 8),
        ("Sep", 9), ("Oct", 10), ("Nov", 11), ("Dec", 12)
    ]

    records = []

    for m_name, m_num in months:
        X_pred = pd.DataFrame([{
            "vehicle_type": vtype,
            "brand": br,
            "model": mdl,
            "fuel_type": fuel,
            "location": loc,
            "year": target_year,
            "month": m_num
        }])

        ex_price = models["ex_showroom_price"].predict(X_pred)[0]
        on_price = models["on_road_price"].predict(X_pred)[0]
        disc = models["discount_percentage"].predict(X_pred)[0]
        final_price = on_price * (1 - disc / 100)

        records.append({
            "Month-Year": f"{m_name}-{target_year}",
            "ExShowroom": ex_price,
            "OnRoad": on_price,
            "Discount": disc,
            "FinalPrice": final_price
        })

    pred_df = pd.DataFrame(records)

    # --- Visualization ---
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=pred_df["Month-Year"], y=pred_df["ExShowroom"],
        mode="lines+markers", name="Ex-Showroom Price"
    ))
    fig.add_trace(go.Scatter(
        x=pred_df["Month-Year"], y=pred_df["OnRoad"],
        mode="lines+markers", name="On-Road Price"
    ))
    fig.add_trace(go.Scatter(
        x=pred_df["Month-Year"], y=pred_df["FinalPrice"],
        mode="lines+markers", name="Final Price (After Discount)",
        line=dict(width=3, dash="dot")
    ))

    fig.update_layout(
        title=f"📊 Vehicle Price Forecast ({target_year})",
        template="plotly_dark",
        xaxis_title="Month-Year",
        yaxis_title="Price (₹)",
        hovermode="x unified",
        plot_bgcolor="#0f1117",
        paper_bgcolor="#0f1117",
        font=dict(color="white")
    )

    avg_ex = pred_df["ExShowroom"].mean()
    avg_on = pred_df["OnRoad"].mean()
    avg_disc = pred_df["Discount"].mean()
    avg_final = pred_df["FinalPrice"].mean()
    best_month = pred_df.loc[pred_df["FinalPrice"].idxmin(), "Month-Year"]

    summary = f"""
    ###  **Vehicle Price Forecast Summary ({target_year})**

    | **Detail** | **Value** |
    |-------------|------------|
    |  **Forecast Year** | {target_year} |
    |  **Your Budget** | ₹{int(budget):,} |
    | **Best Time to Buy** | {best_month} |
    | **Average Ex-Showroom** | ₹{int(avg_ex):,} |
    | **Average On-Road** | ₹{int(avg_on):,} |
    |  **Average Discount** | {avg_disc:.2f}% |
    | **Final Avg Price** | ₹{int(avg_final):,} |
    """

    return summary, fig


In [13]:

# Gradio Interface
inputs = [
    gr.Dropdown(original_values["vehicle_type"], label="Vehicle Type"),
    gr.Dropdown(original_values["brand"], label="Brand"),
    gr.Dropdown(original_values["model"], label="Model"),
    gr.Dropdown(original_values["fuel_type"], label="Fuel Type"),
    gr.Dropdown(original_values["location"], label="Location"),
    gr.Number(label="Your Budget (₹)", value=500000),
    gr.Number(label="Year to Predict", value=2026)   # User can enter ANY year
]

outputs = [
    gr.Markdown(label="Forecast Summary"),
    gr.Plot(label="Forecast Chart")
]

gr.Interface(
    fn=predict_prices,
    inputs=inputs,
    outputs=outputs,
    title="Vehicle Price Forecast Predictor",
    description="Predict Ex-showroom, On-road, Discount %, and Final Prices for ANY selected year using Random Forest Regression.",
    theme="gradio/soft",
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20346e270f28bad5b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
